# Checkpoint C — Temporal integrity (C18–C24)

**Scope:** technical chronology and observability of the original Turno extracts, **not** a payment-success KPI or an experimental effect estimate. Source strings remain immutable; all derived datetimes are timezone-aware UTC. Prior Checkpoints A and B identify status/capture disagreements; no hierarchy of evidence is selected here.

**Reproduction:** from the project root or this notebook folder, install the Python 3.13 dependencies from `pyproject.toml`, place the three original CSV files unchanged in `data/raw/`, select the local Python 3.13 Jupyter kernel, and run Restart & Run All. Every C18–C24 heading includes a procedure, validation, limitations and technical status. V7 must not be declared passed without a successful full clean-kernel run. Calendar-day follow-up endpoints are **diagnostic assumptions**, not proven timestamps of extraction.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
start = Path.cwd().resolve()
ROOT = next((v for v in [start, *start.parents] if (v/'pyproject.toml').is_file() and (v/'data/raw').is_dir()), None)
if ROOT is None: raise FileNotFoundError('pyproject.toml and data/raw/ must exist in project root')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.foundation_a import load_raw, assert_conserved
from src.foundation_b import status_capture_matrix
from src.foundation_ce import utc_timestamp as ts, parse_utc, compare_times, potential_window_coverage
frames, prov = load_raw(ROOT)
e,a,p = frames['exposures'],frames['assignments'],frames['payments']
print('Python:',sys.version.split()[0], 'Pandas:',pd.__version__)
display(prov)
assert (len(e),len(a),len(p))==(14708,279847,306957), 'Source population changed; review before relying on prior checkpoints'
assert all(not f[col].isna().any() and not f[col].duplicated().any() for f,col in ((e,'host_id'),(a,'assignment_id'),(p,'payment_log_id')))
assert not (set(a.host_id)-set(e.host_id)) and not(set(p.assignment_id)-set(a.assignment_id))
assert (p.merge(a[['assignment_id','host_id']],on=['assignment_id','host_id'],how='left',validate='many_to_one').shape[0]==len(p))
PSTART,PEND = ts('2025-08-18'),ts('2025-11-20')
BASE, INC0, INC1 = ts('2025-07-18'),ts('2025-09-17'),ts('2025-10-01')
EXTEND = ts('2025-12-20')
ea=e.assign(first_exposure_utc=parse_utc(e.first_exposed_at))
aa=a.assign(booked_utc=parse_utc(a.created_at), completed_utc=parse_utc(a.completed_at))
pp=p.assign(log_created_utc=parse_utc(p.created_at), capture_utc=parse_utc(p.captured_at))

Python: 3.13.5 Pandas: 2.2.3


,dataset,file,bytes,sha256,columns_unexpected
0,exposures,experiment_exposures.csv,481724,9cce67cd8d90d292c0703416b3c28c69783c56d10c2f99...,
1,assignments,assignments.csv,16411731,c66a6d5cb89b62bc3542ee9184e75c8ded41589e91dac9...,
2,payments,payment_logs.csv,24857672,ab6bf97f2ca216504058b5de1deb2f80641d783e8e2365...,


## C18 — Actual observed coverage versus documented boundaries
**Procedure:** display observed min/max UTC, missing and invalid source timestamps. **Validation:** `valid + absent + invalid = rows`. Compare against documented baseline, experiment, and follow-up without pretending that maximum observed event time proves the extraction's final observation instant. **Status: technically answerable for observed values; extraction completeness unknown.**

In [2]:
fields=[('exposures','first_exposed_at',ea.first_exposure_utc,e.first_exposed_at),('assignments','created_at',aa.booked_utc,a.created_at),('assignments','completed_at',aa.completed_utc,a.completed_at),('payment_logs','created_at',pp.log_created_utc,p.created_at),('payment_logs','captured_at',pp.capture_utc,p.captured_at)]
records=[]
for name,col,derived,raw in fields:
    nvalid=int(derived.notna().sum()); null=int(raw.isna().sum()); bad=int((raw.notna()&derived.isna()).sum())
    assert_conserved(len(raw),nvalid,null,bad)
    records.append(dict(dataset=name,field=col,rows=len(raw),valid=nvalid,missing=null,invalid=bad,observed_min_utc=derived.min(),observed_max_utc=derived.max()))
coverage=pd.DataFrame(records);display(coverage)
period_check=pd.DataFrame([{'item':'completed projects before baseline','rows':int(aa.completed_utc.lt(BASE).sum())},{'item':'completions outside documented baseline+experiment','rows':int((aa.completed_utc.lt(BASE)|aa.completed_utc.ge(PEND)).sum())},{'item':'exposures outside experiment date bounds','rows':int((ea.first_exposure_utc.lt(PSTART)|ea.first_exposure_utc.ge(PEND)).sum())},{'item':'captures on/after 20 December','rows':int(pp.capture_utc.ge(EXTEND).sum())},{'item':'payment records created on/after 20 December','rows':int(pp.log_created_utc.ge(EXTEND).sum())}]);display(period_check)
print('Observed maximum is NOT the extraction cutoff; records created before 2025 do not prove timestamp errors.')

,dataset,field,rows,valid,missing,invalid,observed_min_utc,observed_max_utc
0,exposures,first_exposed_at,14708,14708,0,0,2025-08-18 13:15:07+00:00,2025-11-19 14:45:45+00:00
1,assignments,created_at,279847,279847,0,0,2022-02-13 01:02:12+00:00,2025-11-19 21:25:49+00:00
2,assignments,completed_at,279847,279847,0,0,2025-07-18 00:02:24+00:00,2025-11-19 23:59:58+00:00
3,payment_logs,created_at,306957,306957,0,0,2022-02-13 01:17:01+00:00,2025-12-19 19:25:56+00:00
4,payment_logs,captured_at,306957,279285,27672,0,2024-10-08 16:56:49+00:00,2025-12-19 19:25:56+00:00


,item,rows
0,completed projects before baseline,0
1,completions outside documented baseline+experi...,0
2,exposures outside experiment date bounds,0
3,captures on/after 20 December,0
4,payment records created on/after 20 December,0


Observed maximum is NOT the extraction cutoff; records created before 2025 do not prove timestamp errors.


## C19 — Chronological inconsistencies and unusual sequences
**Procedure:** count invalid timestamps, negative project durations and payment capture before log creation. **Validation:** complementary chronological buckets reconcile. **Limitations:** log creation can legitimately precede project completion, and capture before completion is not automatically invalid; this extract has no event-level transaction authorization type. **Status: answered for recorded chronology.**

In [3]:
negative_booking = aa.completed_utc.lt(aa.booked_utc)
log_capture_relation=compare_times(pp.capture_utc,pp.log_created_utc)
checks=pd.DataFrame([{'check':'completed before booked','attempt_or_project_rows':int(negative_booking.sum()),'grain':'project'},{'check':'capture before payment record created','attempt_or_project_rows':int(log_capture_relation.eq('before').sum()),'grain':'attempt'},{'check':'capture exactly at payment record creation','attempt_or_project_rows':int(log_capture_relation.eq('same_instant').sum()),'grain':'attempt'},{'check':'capture after payment record creation','attempt_or_project_rows':int(log_capture_relation.eq('after').sum()),'grain':'attempt'},{'check':'capture missing or unable to parse','attempt_or_project_rows':int(log_capture_relation.eq('unknown').sum()),'grain':'attempt'}]);display(checks)
assert_conserved(len(pp), *[int(log_capture_relation.eq(x).sum()) for x in ['before','same_instant','after','unknown']])
if negative_booking.any():display(aa.loc[negative_booking,['assignment_id','created_at','completed_at']].head(10))
if log_capture_relation.eq('before').any():display(pp.loc[log_capture_relation.eq('before'),['payment_log_id','created_at','captured_at']].head(8))

,check,attempt_or_project_rows,grain
0,completed before booked,9,project
1,capture before payment record created,1821,attempt
2,capture exactly at payment record creation,3175,attempt
3,capture after payment record creation,274289,attempt
4,capture missing or unable to parse,27672,attempt


,assignment_id,created_at,completed_at
163678,163679,2025-09-01 01:49:09,2025-09-01 01:24:00
188594,188595,2025-09-14 14:59:50,2025-09-14 14:00:00
191804,191805,2025-09-16 01:23:58,2025-09-15 22:00:00
229465,229466,2025-10-06 17:42:13,2025-10-06 12:30:00
229506,229507,2025-10-06 18:05:22,2025-10-06 18:00:00
243853,243854,2025-10-15 14:59:25,2025-10-15 14:30:00
253019,253020,2025-10-21 19:35:45,2025-10-21 16:35:00
265208,265209,2025-10-31 14:14:54,2025-10-31 12:30:00
266292,266293,2025-11-01 14:35:12,2025-11-01 13:40:00


,payment_log_id,created_at,captured_at
4043,4044,2025-07-18 07:52:14,2025-07-18 07:52:13
5620,5621,2025-07-18 15:58:05,2025-07-18 15:58:04
5763,5764,2025-07-18 19:47:25,2025-07-18 19:47:24
6485,6486,2025-07-19 07:36:36,2025-07-19 07:36:35
10763,10764,2025-07-19 17:08:58,2025-07-19 17:08:57
10799,10800,2025-07-19 18:20:08,2025-07-19 18:20:07
10846,10847,2025-07-19 19:46:07,2025-07-19 19:46:04
10851,10852,2025-07-19 20:10:15,2025-07-19 20:10:14


## C20 — Payment-log creation relative to project completion
**Procedure:** one-to-many logs → unique assignment and host via guarded join; classify timestamps and elapsed hours, grouped by method/group only descriptively. **Validation:** preserve all 306,957 attempts; time buckets are exhaustive. **Limitation:** the time a payment record was created is **not** the time of its actual failure, attempted charge, or CS alert. **Status: answered descriptively.**

In [4]:
ap=aa[['assignment_id','host_id','completed_utc']].rename(columns={'host_id':'project_host'})
pa=pp.merge(ap,on='assignment_id',how='left',validate='many_to_one',indicator=True)
assert len(pa)==len(pp) and pa['_merge'].eq('both').all() and pa.host_id.eq(pa.project_host).all()
pa=pa.drop(columns='_merge')
pa['relative_creation']=compare_times(pa.log_created_utc,pa.completed_utc)
pa['hours_created_minus_completion']=(pa.log_created_utc-pa.completed_utc).dt.total_seconds()/3600
pa=pa.merge(e[['host_id','experiment_group']],on='host_id',validate='many_to_one',how='left')
assert len(pa)==len(pp) and pa.experiment_group.notna().all()
creation_buckets=pa.relative_creation.value_counts(dropna=False).rename_axis('relationship').reset_index(name='payment_attempts');display(creation_buckets)
assert_conserved(len(pa),*creation_buckets.payment_attempts)
display(pa.groupby(['experiment_group','payment_method','relative_creation'],dropna=False).size().rename('attempts').reset_index().sort_values('attempts',ascending=False).head(30))
display(pa.hours_created_minus_completion.describe(percentiles=[.01,.1,.5,.9,.99]).to_frame('hours'))

,relationship,payment_attempts
0,before,284113
1,after,22843
2,same_instant,1


,experiment_group,payment_method,relative_creation,attempts
4,control,card,before,94035
10,test,card,before,93693
1,control,bank_account,before,48357
8,test,bank_account,before,47582
3,control,card,after,10041
9,test,card,after,9262
0,control,bank_account,after,1721
7,test,bank_account,after,1632
12,test,credits,before,265
6,control,credits,before,181


,hours
count,306957.000000
mean,-39.899926
std,252.472665
min,-32246.632778
1%,-170.986422
10%,-61.227500
50%,-34.678889
90%,-7.551556
99%,93.581489
max,3522.046667


## C21 — Timestamped captures relative to completion
**Procedure:** consider only records with **parseable** `captured_at`; classify before/equal/after at the attempt grain, then count distinct projects per bucket (overlapping categories). Compare contradictory status rows with B12. **Validation:** observed timestamp buckets reconcile to the number of valid timestamped attempts. **Limitation:** timestamped attempts are not proof of full settlement; project buckets can overlap if a project has more than one captured timestamp. **Status: answered for observed timestamp evidence.**

In [5]:
pa['capture_relative_completion']=compare_times(pa.capture_utc,pa.completed_utc)
capcat=pa.capture_relative_completion.value_counts().rename_axis('when').reset_index(name='attempts');display(capcat)
assert_conserved(len(pa),*capcat.attempts)
display(pa.groupby(['experiment_group','payment_method','capture_relative_completion'],dropna=False).agg(attempts=('payment_log_id','size'),distinct_projects=('assignment_id','nunique')).reset_index().sort_values('attempts',ascending=False).head(40))
print('Distinct projects in overlapping time buckets:',pa.loc[pa.capture_utc.notna()].groupby('capture_relative_completion').assignment_id.nunique().to_dict())
pa['status_capture_disagreement']=(pa.payment_status.eq('succeeded') ^ pa.captured_at.notna()).fillna(False)
print('Discordant attempts, from B12 rules:',int(pa.status_capture_disagreement.sum()))
assert int(pa.status_capture_disagreement.sum())==2316

,when,attempts
0,after,279129
1,unknown,27672
2,before,156


,experiment_group,payment_method,capture_relative_completion,attempts,distinct_projects
11,test,card,after,93993,93846
3,control,card,after,90496,90460
8,test,bank_account,after,47562,47447
0,control,bank_account,after,46539,46342
5,control,card,unknown,13508,6755
13,test,card,unknown,8907,5167
2,control,bank_account,unknown,3524,1496
10,test,bank_account,unknown,1639,1477
14,test,credits,after,276,276
6,control,credits,after,263,263


Distinct projects in overlapping time buckets: {'after': 278438, 'before': 156}
Discordant attempts, from B12 rules: 2316


### C21/C24 follow-up — UTC distribution of status/capture disagreements

**Procedure:** use the **attempt-grain** B12 disagreement flag (`payment_status == succeeded` XOR non-null raw `captured_at`), then attach each attempt to the project's unique recorded completion timestamp and the recorded experiment group. Show two **separate** tables: (1) project **completion UTC** period and (2) payment-log **creation UTC** period. The second is a record-creation diagnostic **not** a timestamp of charge failure, CS alert or recovery. All UTC periods are half-open and disjoint: before baseline `< 2025-07-18`; baseline `[2025-07-18, 2025-08-18)`; experiment before CS incident `[2025-08-18, 2025-09-17)`; CS incident `[2025-09-17, 2025-10-01)`; experiment after CS incident `[2025-10-01, 2025-11-20)`; after experiment `>= 2025-11-20`. Unknown times remain a separate category.

**Validation:** the **attempts** in each table sum to the B12 universe of 2,316, and the distinct affected **projects** across the full discordant-attempt extract sum to 2,075. Project counts are additive **only** for the completion-based table, because a project has one recorded completion time; creation-period distinct-project counts **may overlap across periods**. Explicit boundary tests and independent comparisons to the original B12 source frame are included. **Status:** observed temporal distribution only; no causal attribution to CS and no definitive financial-capture definition.

In [6]:
# The status/capture discrepancy definition is inherited from B12; no record is discarded.
b12_raw_disagreement = p.payment_status.eq('succeeded') ^ p.captured_at.notna()
discordant = pa.loc[pa.status_capture_disagreement].copy()
assert int(b12_raw_disagreement.sum()) == len(discordant) == 2316
assert int(p.loc[b12_raw_disagreement, 'assignment_id'].nunique()) == int(discordant.assignment_id.nunique()) == 2075
assert discordant.payment_log_id.is_unique and discordant.completed_utc.notna().all()

UTC_PERIODS = ('before_baseline', 'baseline', 'experiment_pre_incident',
               'CS_incident', 'experiment_post_incident', 'after_experiment', 'unknown_timestamp')
def disjoint_utc_period(timestamp_series):
    """Classify UTC event timestamps into exhaustive, non-overlapping diagnostic periods."""
    labels = pd.Series('unknown_timestamp', index=timestamp_series.index, dtype='string')
    labels.loc[timestamp_series.lt(BASE)] = 'before_baseline'
    labels.loc[timestamp_series.ge(BASE) & timestamp_series.lt(PSTART)] = 'baseline'
    labels.loc[timestamp_series.ge(PSTART) & timestamp_series.lt(INC0)] = 'experiment_pre_incident'
    labels.loc[timestamp_series.ge(INC0) & timestamp_series.lt(INC1)] = 'CS_incident'
    labels.loc[timestamp_series.ge(INC1) & timestamp_series.lt(PEND)] = 'experiment_post_incident'
    labels.loc[timestamp_series.ge(PEND)] = 'after_experiment'
    assert len(labels) == sum(int(labels.eq(x).sum()) for x in UTC_PERIODS)
    return labels

# Explicit tests of both sides of every half-open UTC boundary and missing timestamps.
boundary_input = pd.Series([BASE - pd.Timedelta(seconds=1), BASE, PSTART - pd.Timedelta(seconds=1),
    PSTART, INC0 - pd.Timedelta(seconds=1), INC0, INC1 - pd.Timedelta(seconds=1),
    INC1, PEND - pd.Timedelta(seconds=1), PEND, pd.NaT])
boundary_expected = ['before_baseline', 'baseline', 'baseline', 'experiment_pre_incident',
    'experiment_pre_incident', 'CS_incident', 'CS_incident', 'experiment_post_incident',
    'experiment_post_incident', 'after_experiment', 'unknown_timestamp']
assert disjoint_utc_period(boundary_input).tolist() == boundary_expected

discordant['completion_period_utc'] = disjoint_utc_period(discordant.completed_utc)
discordant['log_creation_period_utc'] = disjoint_utc_period(discordant.log_created_utc)
index = pd.MultiIndex.from_product([sorted(e.experiment_group.unique()), UTC_PERIODS],
    names=['experiment_group', 'period_utc'])
def discordance_period_table(period_col):
    table = discordant.groupby(['experiment_group', period_col], dropna=False).agg(
        discordant_attempts=('payment_log_id', 'size'),
        distinct_projects_within_period=('assignment_id', 'nunique'))
    table.index = table.index.set_names(['experiment_group', 'period_utc'])
    return table.reindex(index, fill_value=0).reset_index()

completion_discordance_table = discordance_period_table('completion_period_utc')
creation_discordance_table = discordance_period_table('log_creation_period_utc')
all_discordant_projects = int(discordant.assignment_id.nunique())
for table in (completion_discordance_table, creation_discordance_table):
    assert int(table.discordant_attempts.sum()) == len(discordant)
    assert table.discordant_attempts.ge(table.distinct_projects_within_period).all()
assert int(completion_discordance_table.distinct_projects_within_period.sum()) == all_discordant_projects
# The sum of log-creation period project memberships may exceed the unique project universe.
creation_memberships = int(creation_discordance_table.distinct_projects_within_period.sum())
assert creation_memberships >= all_discordant_projects

print('C21/C24: DISCORDANT ATTEMPTS BY PROJECT COMPLETION TIME (UTC); distinct projects additive across rows')
display(completion_discordance_table)
print('C21/C24: DISCORDANT ATTEMPTS BY PAYMENT LOG CREATION TIME (UTC); distinct projects may overlap across periods')
display(creation_discordance_table)
print('Reconciliation: discordant attempts =', len(discordant),
      '; unique affected projects =', all_discordant_projects,
      '; log-creation category project memberships =', creation_memberships,
      '; overlapping memberships beyond unique projects =', creation_memberships - all_discordant_projects)
print('This periodization does NOT locate failure times, CS alerts, causal incident exposure or definitive payment captures.')

C21/C24: DISCORDANT ATTEMPTS BY PROJECT COMPLETION TIME (UTC); distinct projects additive across rows


,experiment_group,period_utc,discordant_attempts,distinct_projects_within_period
0,control,before_baseline,0,0
1,control,baseline,271,244
2,control,experiment_pre_incident,268,238
3,control,CS_incident,112,100
4,control,experiment_post_incident,484,417
5,control,after_experiment,0,0
6,control,unknown_timestamp,0,0
7,test,before_baseline,0,0
8,test,baseline,285,259
9,test,experiment_pre_incident,349,321


C21/C24: DISCORDANT ATTEMPTS BY PAYMENT LOG CREATION TIME (UTC); distinct projects may overlap across periods


,experiment_group,period_utc,discordant_attempts,distinct_projects_within_period
0,control,before_baseline,13,13
1,control,baseline,275,256
2,control,experiment_pre_incident,260,237
3,control,CS_incident,126,117
4,control,experiment_post_incident,443,396
5,control,after_experiment,18,14
6,control,unknown_timestamp,0,0
7,test,before_baseline,15,15
8,test,baseline,280,261
9,test,experiment_pre_incident,340,316


Reconciliation: discordant attempts = 2316 ; unique affected projects = 2075 ; log-creation category project memberships = 2123 ; overlapping memberships beyond unique projects = 48
This periodization does NOT locate failure times, CS alerts, causal incident exposure or definitive payment captures.


## C22 — Booked and completed relative to recorded first exposure
**Procedure:** left join all completed projects to host exposure records (many-to-one); classify project creation and completion independently. **Validation:** chronological cross-tab sums to 279,847 projects with no silent exclusion. **Limitation:** `first_exposed_at` is neither verified randomization instant nor technical application timestamp. **Status: answered for recorded timestamps.**

In [7]:
ae=aa.merge(ea[['host_id','experiment_group','first_exposure_utc']],on='host_id',how='left',validate='many_to_one')
assert len(ae)==len(aa) and ae.first_exposure_utc.notna().all()
ae['creation_vs_exposure']=compare_times(ae.booked_utc,ae.first_exposure_utc)
ae['completion_vs_exposure']=compare_times(ae.completed_utc,ae.first_exposure_utc)
exposure_matrix=pd.crosstab(ae.creation_vs_exposure,ae.completion_vs_exposure,dropna=False);display(exposure_matrix)
assert_conserved(len(ae),int(exposure_matrix.to_numpy().sum()))
print('Created before first recorded exposure but completed after:', int((ae.creation_vs_exposure.eq('before') & ae.completion_vs_exposure.eq('after')).sum()))
display(ae.groupby(['experiment_group','creation_vs_exposure','completion_vs_exposure'],dropna=False).size().rename('projects').reset_index())
# The experiment-date subset is a distinct universe from the full extract.
in_experiment=ae.completed_utc.ge(PSTART)&ae.completed_utc.lt(PEND)
preexposure_during_experiment=int((in_experiment & ae.completion_vs_exposure.eq('before')).sum())
assert_conserved(int(in_experiment.sum()), preexposure_during_experiment, int((in_experiment & ae.completion_vs_exposure.eq('after')).sum()),int((in_experiment & ae.completion_vs_exposure.eq('same_instant')).sum()))
print('Experiment-period projects completed before first recorded exposure (not proven treatment exposure):',preexposure_during_experiment)

completion_vs_exposure,after,before
creation_vs_exposure,,
after,130917,0
before,65294,83636


Created before first recorded exposure but completed after: 65294


,experiment_group,creation_vs_exposure,completion_vs_exposure,projects
0,control,after,after,63983
1,control,before,after,32299
2,control,before,before,41304
3,test,after,after,66934
4,test,before,after,32995
5,test,before,before,42332


Experiment-period projects completed before first recorded exposure (not proven treatment exposure): 5091


### C22 follow-up — Reproducible pre-exposure completion reconciliation

**Procedure:** restrict to projects whose `completed_at` lies within the **experiment completion window** `[2025-08-18 00:00 UTC, 2025-11-20 00:00 UTC)`, then select completions **strictly earlier than** the corresponding host's recorded `first_exposed_at`. Group those **projects** by recorded experiment group. Display both observed values and the previously reported reference numbers solely for comparison — all reconciliation assertions use data-derived counts, **not fixed reference values**. Independently verify the filter using the unique host lookup rather than the joined working table.

**Validation:** group sums equal the actual filtered project count; experiment-window membership, timestamp ordering and uniqueness are asserted. **Limitation:** a completion before first recorded exposure is not evidence of invalid randomization or a reason to exclude the project; no eligibility or experimental-population rule is imposed.

In [8]:
# Derive the exact experiment-completion, strictly pre-recorded-exposure universe.
c22_experiment_mask = ae.completed_utc.ge(PSTART) & ae.completed_utc.lt(PEND)
c22_pre_exposure_mask = (c22_experiment_mask & ae.first_exposure_utc.notna()
                         & ae.completed_utc.lt(ae.first_exposure_utc))
assert ae.assignment_id.is_unique and ae.loc[c22_pre_exposure_mask, 'assignment_id'].is_unique
assert c22_pre_exposure_mask.le(c22_experiment_mask).all()
assert ae.loc[c22_pre_exposure_mask, 'completed_utc'].lt(
    ae.loc[c22_pre_exposure_mask, 'first_exposure_utc']).all()
assert int(c22_experiment_mask.sum()) == int(in_experiment.sum())

c22_by_group = (ae.loc[c22_pre_exposure_mask].groupby('experiment_group', dropna=False)
                .assignment_id.nunique().reindex(sorted(e.experiment_group.unique()), fill_value=0)
                .rename('recalculated_projects'))
c22_actual_total = int(c22_pre_exposure_mask.sum())
assert int(c22_by_group.sum()) == c22_actual_total
assert c22_actual_total == int(ae.loc[c22_pre_exposure_mask, 'assignment_id'].nunique())

# Independent reconstruction from assignments + keyed exposures, without reusing the ae join.
exposure_by_host = ea.set_index('host_id').first_exposure_utc
independent_mask = (aa.completed_utc.ge(PSTART) & aa.completed_utc.lt(PEND)
                    & aa.completed_utc.lt(aa.host_id.map(exposure_by_host)))
assert int(independent_mask.sum()) == c22_actual_total
independent_by_group = (aa.loc[independent_mask]
    .assign(experiment_group=lambda x: x.host_id.map(ea.set_index('host_id').experiment_group))
    .groupby('experiment_group', dropna=False).assignment_id.nunique())
assert c22_by_group.equals(independent_by_group.reindex(c22_by_group.index, fill_value=0).rename('recalculated_projects'))

# Fixed reported values are comparison-only, never the operands of a reconciliation assertion.
previously_reported = pd.Series({'control': 2573, 'test': 2518}, name='previously_reported_projects')
c22_check = c22_by_group.to_frame().join(previously_reported, how='outer')
c22_check['difference_vs_reported'] = c22_check.recalculated_projects - c22_check.previously_reported_projects
c22_display = pd.concat([c22_check, pd.DataFrame([{
    'recalculated_projects': c22_actual_total,
    'previously_reported_projects': int(previously_reported.sum()),
    'difference_vs_reported': c22_actual_total - int(previously_reported.sum())}], index=['TOTAL'])])
print('C22: EXPERIMENT-PERIOD PROJECTS COMPLETED STRICTLY BEFORE FIRST RECORDED EXPOSURE (UTC)')
display(c22_display)
print('Experiment-completion projects:', int(c22_experiment_mask.sum()),
      '; pre-exposure subset:', c22_actual_total,
      '; group sum:', int(c22_by_group.sum()), '; independent-source check: PASS')
if c22_check.difference_vs_reported.fillna(0).ne(0).any() or c22_check.isna().any().any():
    print('ATTENTION: observed group totals differ from the previous reported decomposition.')
else:
    print('The data-derived decomposition matches the previously reported group counts.')

C22: EXPERIMENT-PERIOD PROJECTS COMPLETED STRICTLY BEFORE FIRST RECORDED EXPOSURE (UTC)


,recalculated_projects,previously_reported_projects,difference_vs_reported
control,2573,2573,0
test,2518,2518,0
TOTAL,5091,5091,0


Experiment-completion projects: 201302 ; pre-exposure subset: 5091 ; group sum: 5091 ; independent-source check: PASS
The data-derived decomposition matches the previously reported group counts.


## C23 — Candidate follow-up windows and ambiguous extraction endpoint
**Procedure:** model elapsed time D0/D1/D7/D14/D30 from exact completion instants. Because the brief specifies follow-up **through 2025-12-19** without an exact UTC end time, give conservative covered projects (`deadline` strictly before Dec 19 00:00), last-day uncertain (`deadline` falls on Dec 19), and beyond horizon. Separate presence of at least one recorded timestamped capture by deadline (including precompletion captures) and a capture recorded *between* completion and deadline. **Validation:** every window's coverage categories sum to all completed projects, independent direct-time bound check, and captured-by-deadline totals never exceed project counts. **Caveat:** the conservative category assumes documented continuous event logging through Dec 19 00:00; actual completeness cannot be proven from observed maximum event timestamps. These are **candidate boundaries**, not payment outcome KPIs. **Status: partially answerable — exact completeness and full settlement not provable.**

In [9]:
# Timestamped captures only: separate precompletion evidence from within-window captures.
valid_cap=pa.loc[pa.capture_utc.notna(),['assignment_id','capture_utc','completed_utc']].copy()
valid_cap['post_completion']=valid_cap.capture_utc.ge(valid_cap.completed_utc)
first_any=valid_cap.groupby('assignment_id').capture_utc.min()
first_post=valid_cap.loc[valid_cap.post_completion].groupby('assignment_id').capture_utc.min()
aa['first_any_capture']=aa.assignment_id.map(first_any)
aa['first_post_completion_capture']=aa.assignment_id.map(first_post)
rows=[]
for days in (0,1,7,14,30):
    coverage_type=potential_window_coverage(aa.completed_utc,days)
    counts=coverage_type.value_counts(dropna=False).to_dict()
    deadline=aa.completed_utc+pd.Timedelta(days=days)
    by_deadline=aa.first_any_capture.notna() & aa.first_any_capture.le(deadline)
    within_window=aa.first_post_completion_capture.notna() & aa.first_post_completion_capture.le(deadline)
    assert_conserved(len(aa),*[int(counts.get(cat,0)) for cat in ['within_documented_horizon_conservative','last_day_time_uncertain','beyond_documented_horizon','unknown_completion']])
    independent=int((deadline < ts('2025-12-19')).sum())
    assert independent==int(counts.get('within_documented_horizon_conservative',0))
    assert int(within_window.sum())<=int(by_deadline.sum())<=len(aa)
    rows.append({'window':f'D{days}','elapsed_days':days,'total_projects':len(aa),'conservative_horizon_coverage':counts.get('within_documented_horizon_conservative',0),'last_day_uncertain':counts.get('last_day_time_uncertain',0),'beyond_horizon':counts.get('beyond_documented_horizon',0),'unknown_completion':counts.get('unknown_completion',0),'projects_with_timestamped_capture_by_deadline_including_early':int(by_deadline.sum()),'projects_with_timestamped_capture_in_0_to_deadline':int(within_window.sum()),'no_timestamp_by_deadline_NOT_failure':int((~by_deadline).sum())})
followup=pd.DataFrame(rows);display(followup)
print('Observed latest timestamp:',pp.capture_utc.max(),'— not evidence of extraction completeness')
print('Source statuses with succeeded but no timestamp:',int((p.payment_status.eq('succeeded')&p.captured_at.isna()).sum()))

,window,elapsed_days,total_projects,conservative_horizon_coverage,last_day_uncertain,beyond_horizon,unknown_completion,projects_with_timestamped_capture_by_deadline_including_early,projects_with_timestamped_capture_in_0_to_deadline,no_timestamp_by_deadline_NOT_failure
0,D0,0,279847,279847,0,0,0,156,0,279691
1,D1,1,279847,279847,0,0,0,275697,275576,4150
2,D7,7,279847,279847,0,0,0,277977,277856,1870
3,D14,14,279847,279847,0,0,0,278279,278158,1568
4,D30,30,279847,278595,1252,0,0,278471,278350,1376


Observed latest timestamp: 2025-12-19 19:25:56+00:00 — not evidence of extraction completeness
Source statuses with succeeded but no timestamp: 239


## C24 — September CS incident: what can be located in time?
**Procedure:** separately tabulate project completions, log-creation timestamps, captured timestamps and manual-flagged log creation by **experiment group** and before/during/after the reported incident (inclusive UTC dates Sept 17–30). **Validation:** every event series reconciles to its own nonmissing timestamp universe; do not conflate attempts and projects. **Limitations:** `created_at` is not actual failure/alert time; `is_manual_attempt` omits unlogged CS outreach; the dataset has no alert audit trail. Incident-related causal claims are not possible. **Status: partially answerable for observed timestamped events only.**

In [10]:
def incident_bucket(s):
    z=pd.Series('unobserved_time',index=s.index,dtype='string')
    z.loc[s.notna() & s.lt(PSTART)]='before_experiment'
    z.loc[s.ge(PSTART)&s.lt(INC0)]='before_incident_in_experiment'
    z.loc[s.ge(INC0)&s.lt(INC1)]='during_incident'
    z.loc[s.ge(INC1)&s.lt(PEND)]='after_incident_in_experiment'
    z.loc[s.ge(PEND)]='after_experiment'
    return z
pa['inc_created']=incident_bucket(pa.log_created_utc)
pa['inc_capture']=incident_bucket(pa.capture_utc)
ae['inc_completed']=incident_bucket(ae.completed_utc)
source_summary=[]
for event,frame,field,grain in [('project_completion',ae,'inc_completed','project'),('payment_log_created',pa,'inc_created','attempt'),('timestamped_capture',pa.loc[pa.capture_utc.notna()],'inc_capture','attempt'),('manual_log_created',pa.loc[pa.is_manual_attempt.eq('true')],'inc_created','attempt')]:
    x=frame.groupby(['experiment_group',field],dropna=False).size()
    assert int(x.sum())==len(frame)
    source_summary.extend([{'event':event,'grain':grain,'group':i,'period':j,'observations':int(count)} for (i,j),count in x.items()])
incident_table=pd.DataFrame(source_summary)
display(incident_table.pivot_table(index=['event','grain','group'],columns='period',values='observations',aggfunc='sum',fill_value=0))
print('No timestamp records the moment an alert should have been sent; this is not a causal incident analysis.')
# Boundary test: incident is [2025-09-17 00:00 UTC, 2025-10-01 00:00 UTC).
assert incident_bucket(pd.Series([INC0, INC1])).tolist()==['during_incident','after_incident_in_experiment']

period                               after_experiment  ...  during_incident
event               grain   group                      ...                 
manual_log_created  attempt control               125  ...              354
                            test                  112  ...              891
payment_log_created attempt control               130  ...            16865
                            test                  115  ...            16729
project_completion  project control                 0  ...            14974
                            test                    0  ...            15486
timestamped_capture attempt control                75  ...            14906
                            test                   75  ...            15514

[8 rows x 5 columns]

No timestamp records the moment an alert should have been sent; this is not a causal incident analysis.


## C — Technical checkpoint conclusion
Recorded UTC chronologies, censoring bounds and incident-period event timestamps are auditable. **Material dependency for subsequent KPIs:** exact last-observable UTC instant and complete capture-event coverage are not verifiable; D30 near the final completion day is uncertain. Contradictory capture/status evidence also persists. **Checkpoint E may proceed independently** to describe the registered experimental population and recorded exposures, without defining success rates or ITT. **V7 (Checkpoint C) PASSED in a clean Python 3.13.5 Jupyter kernel for this revision; separate local reproduction and historical A/B V7 remain pending.**